In [ ]:
# Persistent paths on Unity Catalog Volumes (survives serverless restarts)
GOLD_PATH = "/Volumes/workspace/legal_data/gold/legal_chunks/"
EMBEDDING_DELTA_PATH = "/Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta"
CHROMA_DB_PATH = "/Volumes/workspace/legal_data/chroma_db/legal_knowledge_test"

COLLECTION_NAME = "legal_knowledge"
PRIMARY_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FALLBACK_EMBED_MODEL = "sentence-transformers/paraphrase-MiniLM-L3-v2"

BATCH_SIZE = 128
FORCE_REBUILD_CHROMA = False  # set True to rebuild collection from scratch


In [ ]:
%pip install -q --upgrade sentence-transformers chromadb transformers accelerate


In [ ]:
import os
import traceback
from datetime import datetime

import chromadb
from sentence_transformers import SentenceTransformer
from pyspark.sql import functions as F


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


In [ ]:
try:
    gold_df = spark.read.format("delta").load(GOLD_PATH)
except Exception as e:
    raise RuntimeError(
        f"Unable to read gold chunks from {GOLD_PATH}. Make sure notebook 03 ran successfully. Error: {e}"
    )

required_cols = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name"]
missing_cols = [c for c in required_cols if c not in gold_df.columns]
if missing_cols:
    raise ValueError(f"Gold table is missing required columns: {missing_cols}")

gold_df = (
    gold_df.select(*required_cols)
    .dropna(subset=["chunk_id", "chunk_text"])
    .dropDuplicates(["chunk_id"])
)

gold_count = gold_df.count()
if gold_count == 0:
    raise ValueError("Gold dataset is empty after filtering. Cannot build embeddings.")

log(f"Gold chunks ready: {gold_count}")
gold_df.show(10, truncate=120)


In [ ]:
embedding_model = None
loaded_model_name = None

for model_name in [PRIMARY_EMBED_MODEL, FALLBACK_EMBED_MODEL]:
    try:
        log(f"Loading embedding model: {model_name}")
        embedding_model = SentenceTransformer(model_name)
        _ = embedding_model.encode(["health check"], show_progress_bar=False)
        loaded_model_name = model_name
        log(f"Embedding model loaded: {model_name}")
        break
    except Exception as e:
        log(f"Failed to load {model_name}: {e}")

if embedding_model is None:
    raise RuntimeError(
        "Could not load any embedding model. Check internet access/Hugging Face access on cluster."
    )


In [ ]:
for p in ["/Volumes/workspace/legal_data/vector_db_test", CHROMA_DB_PATH]:
    os.makedirs(p, exist_ok=True)

client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

if FORCE_REBUILD_CHROMA:
    try:
        client.delete_collection(COLLECTION_NAME)
        log("Deleted existing Chroma collection due to FORCE_REBUILD_CHROMA=True")
    except Exception:
        pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

log(f"Chroma collection ready. Existing vectors: {collection.count()}")


In [ ]:
def safe_str(value):
    if value is None:
        return ""
    return str(value)


def build_metadata(row):
    return {
        "act_name": safe_str(row.act_name),
        "section": safe_str(row.section_number),
        "category": safe_str(row.category),
        "source": safe_str(row.file_name),
    }


In [ ]:
records = []
processed = 0
chroma_upserts = 0
batch_rows = []


def flush_batch(rows):
    texts = []
    ids = []
    metadatas = []

    for r in rows:
        chunk_text = safe_str(r.chunk_text).strip()
        chunk_id = safe_str(r.chunk_id).strip()

        if not chunk_text or not chunk_id:
            continue

        texts.append(chunk_text)
        ids.append(chunk_id)
        metadatas.append(build_metadata(r))

    if not ids:
        return 0, []

    embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()

    ts = datetime.utcnow().isoformat()
    batch_records = []
    for i in range(len(ids)):
        batch_records.append({
            "chunk_id": ids[i],
            "chunk_text": texts[i],
            "act_name": metadatas[i]["act_name"],
            "section_number": metadatas[i]["section"],
            "category": metadatas[i]["category"],
            "file_name": metadatas[i]["source"],
            "embedding": embeddings[i],
            "embedding_model": loaded_model_name,
            "embedding_dim": len(embeddings[i]),
            "updated_at": ts,
        })

    upserted = 0
    try:
        collection.upsert(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas,
        )
        upserted = len(ids)
    except Exception as e:
        log(f"WARNING: Chroma upsert failed for current batch. Delta export will still continue. Error: {e}")

    return upserted, batch_records


for row in gold_df.toLocalIterator():
    batch_rows.append(row)

    if len(batch_rows) >= BATCH_SIZE:
        upserted, batch_records = flush_batch(batch_rows)
        chroma_upserts += upserted
        records.extend(batch_records)
        processed += len(batch_rows)
        log(f"Processed rows: {processed}")
        batch_rows = []

if batch_rows:
    upserted, batch_records = flush_batch(batch_rows)
    chroma_upserts += upserted
    records.extend(batch_records)
    processed += len(batch_rows)

if not records:
    raise RuntimeError("No embedding records were produced.")

embedding_df = spark.createDataFrame(records)
embedding_df = embedding_df.withColumn("updated_at", F.to_timestamp("updated_at"))

(
    embedding_df
    .dropDuplicates(["chunk_id"])
    .write
    .format("delta")
    .mode("overwrite")
    .save(EMBEDDING_DELTA_PATH)
)

log(f"Delta embedding export complete at: {EMBEDDING_DELTA_PATH}")
log(f"Rows written to Delta: {embedding_df.count()}")
log(f"Vectors upserted to Chroma in this run: {chroma_upserts}")
log(f"Current Chroma vector count: {collection.count()}")


In [ ]:
query = "What is the penalty for not wearing a helmet under Indian law?"

query_embedding = embedding_model.encode([query], show_progress_bar=False).tolist()

try:
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=5,
    )

    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]

    print(f"Retrieved docs: {len(docs)}")
    for i, (doc, meta) in enumerate(zip(docs, metas), start=1):
        print(f"\n--- Result {i} ---")
        print(meta)
        print(doc[:400])

except Exception as e:
    print(f"Smoke test query failed: {e}")
    print("Embedding export is still available in Delta at EMBEDDING_DELTA_PATH.")
